# Astra ADAS M5 - Detection Training and Evaluation

This Colab notebook downloads the IDD-D dataset from Kaggle, trains a separate YOLOv8s candidate, compares it with the existing IDD checkpoint on validation data, and exports it without replacing the active local model.

## Get the Kaggle dataset into Colab

The dataset is downloaded directly into Colab's temporary local storage; you do not need to upload it to Google Drive.

1. Open the [Kaggle dataset](https://www.kaggle.com/datasets/redzapdos123/indian-driving-dataset-detections-yolov11), sign in, and accept its terms/access requirements if prompted.
2. In Kaggle, create an API token. In Colab, open **Secrets** (key icon) and add `KAGGLE_USERNAME` and `KAGGLE_KEY` with your Kaggle username and API key. Enable notebook access for both secrets. Never paste the API key into notebook code.
3. Choose a Colab GPU runtime (**Runtime > Change runtime type > T4 GPU**, or another available GPU) and run the notebook cells from top to bottom.
4. Allow roughly 45 GiB of free Colab disk space for the download and extraction. The setup cell checks available space and reports a clear error if there is not enough. Colab local files are temporary and are lost when the runtime is reset.

Google Drive is mounted to keep training outputs and every-epoch model checkpoints persistently, and optionally read `MyDrive/astra_adas/best.pt` for a paired baseline comparison. Checkpoints are saved under `MyDrive/astra_adas_training/runs/<run-name>/weights/` as `epoch*.pt`, plus `last.pt` and `best.pt`. If Kaggle download or Colab disk space is unavailable, the separate Kaggle notebook can use the attached dataset directly.

## Project checklist

- **Prototype path exists:** video-file input, YOLO 15-class detection, ByteTrack IDs, lane/hazard heuristics, relative-depth/motion/TTC telemetry, traffic-light HSV estimates, sign counts, and Streamlit annotated-video/CSV outputs.
- **Detection improvement demonstrated (not retraining):** on the same evenly sampled 200-image IDD-D validation subset, the existing checkpoint's mAP@50 was 0.5045 at 640px and 0.5602 at 960px; Indian-specific mAP@50 was 0.4870 and 0.5194. The 960px CPU path was slower (15.3 vs 29.2 FPS). These are subset validation results, not held-out test results. The notebook also measures validation-stage latency/FPS at batch size 1; this is not end-to-end dashboard FPS.
- **Still partial or missing:** live camera/CARLA input, signal-to-ego-lane association and labeled signal-state evaluation, traffic-sign type recognition, hazard subclasses, calibrated metric depth/speed, tracking ground-truth metrics, validated driving policy/audio alerts, and dashboard feedback/retraining.
- **Training status:** `runs/detect/astra_m5_yolov8s/` contains training arguments and sample plots but no `weights/best.pt` or `results.csv`; a completed M5 run is not verified. The active `models/detection/best.pt` is preserved and must not be overwritten by this notebook. Malformed training annotation rows are omitted from writable label copies; validation/test images with malformed annotations are excluded. The source Kaggle dataset is unchanged and a JSON audit records the exclusions.

## Workflow and evaluation rules

1. Download the dataset from Kaggle in the setup cell; optionally upload the current `models/detection/best.pt` to `MyDrive/astra_adas/best.pt` for paired comparison.
2. Audit all three splits and label IDs; train on **train**, tune/compare on **val** only.
3. Evaluate the existing checkpoint and new candidate at 640px, 768px, and 960px on the same cleaned validation split, with batch size 1 so throughput is more representative of single-video inference. The earlier 200-image results above are a separate measurement.
4. Review validation results before final held-out **test** evaluation. Do not tune from test results. Export a timestamped candidate; manually back up and replace the active local model only after controlled evaluation.

The dataset listing is licensed CC BY-NC-SA 4.0; retain attribution and follow its non-commercial/share-alike terms. Training improves only object detection; it does not complete the other ADAS blocks or establish safety. See [`README.md`](../README.md) for the full implementation checklist and limitations.

In [ ]:
%pip -q install "ultralytics>=8.2.0" pyyaml kaggle

from pathlib import Path
from collections import Counter
from datetime import datetime
import json
import os
import shutil
import subprocess
import yaml
import torch
from ultralytics import YOLO

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training at 960px is intended for a Colab GPU. Dataset checks can run on CPU, but do not start training without enabling a GPU runtime.")

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'astra_adas_training'
BASELINE_WEIGHTS = DRIVE_ROOT / 'astra_adas' / 'best.pt'
KAGGLE_DATASET = 'redzapdos123/indian-driving-dataset-detections-yolov11'
KAGGLE_DOWNLOAD_DIR = Path('/content/idd_kaggle_download')
RUNS_DIR = PROJECT_DIR / 'runs'
EXPORT_DIR = PROJECT_DIR / 'exports'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

KAGGLE_USERNAME = userdata.get('KAGGLE_USERNAME')
KAGGLE_KEY = userdata.get('KAGGLE_KEY')
if not KAGGLE_USERNAME or not KAGGLE_KEY:
    raise ValueError('Add KAGGLE_USERNAME and KAGGLE_KEY in Colab Secrets, then enable notebook access for both.')
os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY'] = KAGGLE_KEY

if not any(KAGGLE_DOWNLOAD_DIR.rglob('train/images')):
    required_free = 45 * 1024**3
    available_free = shutil.disk_usage('/content').free
    if available_free < required_free:
        raise OSError(f'Kaggle download and extraction need about 45 GiB free in Colab storage. Available: {available_free / 1024**3:.1f} GiB. Free space or use the Kaggle notebook instead.')
    KAGGLE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET,
        '-p', str(KAGGLE_DOWNLOAD_DIR), '--unzip'
    ], check=True)
    print('Downloaded Kaggle dataset to:', KAGGLE_DOWNLOAD_DIR)
else:
    print('Using existing Kaggle dataset download:', KAGGLE_DOWNLOAD_DIR)

DATA_DIR = KAGGLE_DOWNLOAD_DIR
print('Kaggle dataset:', KAGGLE_DATASET)
print('Optional paired-comparison baseline:', BASELINE_WEIGHTS)
print('Training output folder:', RUNS_DIR)

In [ ]:
if (DATA_DIR / 'train' / 'images').is_dir():
    SOURCE_DATA_ROOT = DATA_DIR
else:
    roots = [p.parent.parent for p in DATA_DIR.rglob('train/images') if p.is_dir()]
    SOURCE_DATA_ROOT = next((p for p in roots if all((p / split / 'images').is_dir() and (p / split / 'labels').is_dir() for split in ('train', 'val', 'test'))), None)
if SOURCE_DATA_ROOT is None:
    raise FileNotFoundError(f'Could not find train/val/test images and labels inside the Kaggle download at {DATA_DIR}. Check the dataset archive layout.')

source_yamls = list(SOURCE_DATA_ROOT.glob('*.yaml')) + list(SOURCE_DATA_ROOT.glob('*.yml'))
if not source_yamls:
    source_yamls = list(SOURCE_DATA_ROOT.rglob('*.yaml')) + list(SOURCE_DATA_ROOT.rglob('*.yml'))
if not source_yamls:
    raise FileNotFoundError('Dataset data.yaml with class names was not found.')
source_config = yaml.safe_load(source_yamls[0].read_text(encoding='utf-8'))
raw_names = source_config.get('names')
if isinstance(raw_names, dict):
    class_names = [raw_names[i] for i in sorted(raw_names)]
else:
    class_names = list(raw_names or [])
if len(class_names) != 15:
    raise ValueError(f'Expected the project 15-class IDD label map, found {len(class_names)} class names: {class_names}')
if source_config.get('nc', len(class_names)) != len(class_names):
    raise ValueError('Dataset nc does not match the number of class names.')

image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
CLEAN_DATA_ROOT = Path('/content') / ('astra_adas_dataset_clean_' + datetime.now().strftime('%Y%m%d_%H%M%S_%f'))
invalid_rows = []
source_counts = {}
clean_counts = {}
class_counts = Counter()
missing_label_count = {}
excluded_eval_images = {'val': [], 'test': []}

for split in ('train', 'val', 'test'):
    source_images_dir = SOURCE_DATA_ROOT / split / 'images'
    source_labels_dir = SOURCE_DATA_ROOT / split / 'labels'
    clean_images_dir = CLEAN_DATA_ROOT / split / 'images'
    clean_labels_dir = CLEAN_DATA_ROOT / split / 'labels'
    images = [p for p in source_images_dir.rglob('*') if p.is_file() and p.suffix.lower() in image_extensions]
    if not images:
        raise ValueError(f'No images found in {source_images_dir}')
    source_counts[split] = len(images)
    clean_count = 0
    missing = 0

    for image_path in images:
        relative_path = image_path.relative_to(source_images_dir)
        source_label_path = source_labels_dir / relative_path.with_suffix('.txt')
        valid_rows = []
        image_invalid_rows = []
        if source_label_path.is_file():
            for line_number, row in enumerate(source_label_path.read_text(encoding='utf-8').splitlines(), 1):
                if not row.strip():
                    continue
                fields = row.split()
                try:
                    values = [float(value) for value in fields]
                    class_id = int(values[0])
                    valid = (len(values) == 5 and values[0].is_integer() and 0 <= class_id < len(class_names) and all(0.0 <= v <= 1.0 for v in values[1:]) and values[3] > 0.0 and values[4] > 0.0)
                except (ValueError, IndexError):
                    valid = False
                    class_id = -1
                if valid:
                    valid_rows.append(row.strip())
                    if split == 'train':
                        class_counts[class_names[class_id]] += 1
                else:
                    item = {'split': split, 'label_file': str(source_label_path), 'line': line_number, 'row': row}
                    invalid_rows.append(item)
                    image_invalid_rows.append(item)
        else:
            missing += 1

        if image_invalid_rows and split in ('val', 'test'):
            excluded_eval_images[split].append(str(image_path))
            continue

        clean_image_path = clean_images_dir / relative_path
        clean_label_path = clean_labels_dir / relative_path.with_suffix('.txt')
        clean_image_path.parent.mkdir(parents=True, exist_ok=True)
        clean_label_path.parent.mkdir(parents=True, exist_ok=True)
        clean_image_path.symlink_to(image_path.resolve())
        clean_label_path.write_text('\n'.join(valid_rows) + ('\n' if valid_rows else ''), encoding='utf-8')
        clean_count += 1

    clean_counts[split] = clean_count
    missing_label_count[split] = missing

if not clean_counts['val'] or not clean_counts['test']:
    raise ValueError('Cleaning malformed annotations removed every image from validation or test; inspect the audit before training.')

DATA_ROOT = CLEAN_DATA_ROOT
AUDIT_REPORT = EXPORT_DIR / 'dataset_label_audit.json'
AUDIT_REPORT.write_text(json.dumps({
    'source_dataset': str(SOURCE_DATA_ROOT),
    'training_policy': 'Invalid annotation rows are omitted from writable label copies; the Kaggle download is unchanged.',
    'evaluation_policy': 'Validation/test images containing any invalid annotation are excluded to avoid evaluating against incomplete ground truth.',
    'source_image_counts': source_counts,
    'clean_image_counts': clean_counts,
    'missing_label_files': missing_label_count,
    'excluded_validation_test_images': excluded_eval_images,
    'invalid_rows': invalid_rows,
}, indent=2), encoding='utf-8')

print('Source Kaggle dataset:', SOURCE_DATA_ROOT)
print('Writable cleaned dataset view:', DATA_ROOT)
print('Classes:', len(class_names), class_names)
print('Source images:', source_counts)
print('Images used after audit:', clean_counts)
print('Images without label files (treated as negatives):', missing_label_count)
print('Invalid annotation rows:', len(invalid_rows))
print('Evaluation images excluded due to malformed labels:', {k: len(v) for k, v in excluded_eval_images.items()})
print('Training-label instances by class after dropping invalid rows:')
for name in class_names:
    print(f'  {name}: {class_counts[name]}')
print('Audit report:', AUDIT_REPORT)
for item in invalid_rows:
    print(f"  {item['split']} {item['label_file']}:{item['line']}: {item['row']}")

In [ ]:
DATA_YAML = PROJECT_DIR / 'idd_colab.yaml'
data_config = {
    'path': str(DATA_ROOT),
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': class_names,
    'nc': len(class_names),
}
DATA_YAML.write_text(yaml.safe_dump(data_config, sort_keys=False), encoding='utf-8')
print(DATA_YAML.read_text(encoding='utf-8'))


In [ ]:
def normalize_names(names):
    if isinstance(names, dict):
        return [names[i] for i in sorted(names)]
    return list(names)

def evaluate_checkpoint(weights_path, image_sizes=(640, 768, 960), split='val'):
    model = YOLO(str(weights_path))
    model_names = normalize_names(model.names)
    if model_names != class_names:
        raise ValueError(f'Checkpoint class names do not match dataset names. Checkpoint={model_names}; dataset={class_names}')
    measured = {}
    for image_size in image_sizes:
        metrics = model.val(
            data=str(DATA_YAML), split=split, imgsz=image_size, batch=1,
            workers=4, device=0, plots=False, verbose=False,
        )
        box = metrics.box
        speed = metrics.speed
        preprocess_ms = float(speed.get('preprocess', 0.0))
        inference_ms = float(speed.get('inference', 0.0))
        postprocess_ms = float(speed.get('postprocess', 0.0))
        total_ms = preprocess_ms + inference_ms + postprocess_ms
        row = {
            'mAP50': float(box.map50),
            'mAP50_95': float(box.map),
            'validation_batch_size': 1,
            'preprocess_ms_per_image': preprocess_ms,
            'inference_ms_per_image': inference_ms,
            'postprocess_ms_per_image': postprocess_ms,
            'validation_pipeline_fps': 1000.0 / total_ms if total_ms > 0 else None,
            'inference_only_fps': 1000.0 / inference_ms if inference_ms > 0 else None,
        }
        if hasattr(box, 'ap50') and hasattr(box, 'ap_class_index'):
            row['per_class_AP50'] = {
                class_names[int(class_id)]: float(ap)
                for class_id, ap in zip(box.ap_class_index, box.ap50)
            }
        measured[str(image_size)] = row
        print(f"{weights_path.name} | {split} | {image_size}px | mAP50={row['mAP50']:.4f} | mAP50-95={row['mAP50_95']:.4f} | val FPS={row['validation_pipeline_fps']:.1f} (batch=1)")
    return measured

BASELINE_RESULTS = {}
if BASELINE_WEIGHTS.is_file():
    BASELINE_RESULTS = evaluate_checkpoint(BASELINE_WEIGHTS)
else:
    print(f'Baseline comparison skipped: copy the active local checkpoint to {BASELINE_WEIGHTS} and rerun this cell before training if a paired comparison is required.')


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is available. In Colab, choose Runtime > Change runtime type > T4 GPU, then rerun the setup cells.')

EPOCHS = 50
IMAGE_SIZE = 960
BATCH_SIZE = 8  # Reduce to 4 if this GPU runs out of memory.
RUN_NAME = 'astra_m5_yolov8s_960_' + datetime.now().strftime('%Y%m%d_%H%M%S')

model = YOLO('yolov8s.pt')
train_results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    workers=4,
    device=0,
    project=str(RUNS_DIR),
    name=RUN_NAME,
    exist_ok=False,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,
    weight_decay=0.0005,
    mosaic=1.0,
    mixup=0.1,
    degrees=3.0,
    translate=0.1,
    scale=0.5,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    close_mosaic=10,
    patience=20,
    save_period=1,
    plots=True,
    val=True,
    seed=42,
    deterministic=False,
    cache=False,
    amp=True,
    verbose=True,
)

RUN_DIR = Path(train_results.save_dir)
BEST_WEIGHTS = RUN_DIR / 'weights' / 'best.pt'
if not BEST_WEIGHTS.is_file():
    raise FileNotFoundError(f'Training finished without the expected checkpoint: {BEST_WEIGHTS}')
print('Training run:', RUN_DIR)
print('Candidate checkpoint:', BEST_WEIGHTS)
print('Every-epoch checkpoints:', RUN_DIR / 'weights')
print('Training log:', RUN_DIR / 'results.csv')


In [ ]:
CANDIDATE_RESULTS = evaluate_checkpoint(BEST_WEIGHTS)
print('\nValidation comparison (same full val split):')
for image_size, candidate in CANDIDATE_RESULTS.items():
    baseline = BASELINE_RESULTS.get(image_size)
    if baseline:
        delta = candidate['mAP50'] - baseline['mAP50']
        print(f"  {image_size}px: baseline mAP50={baseline['mAP50']:.4f}, candidate={candidate['mAP50']:.4f}, delta={delta:+.4f}; val FPS baseline={baseline['validation_pipeline_fps']:.1f}, candidate={candidate['validation_pipeline_fps']:.1f}")
    else:
        print(f"  {image_size}px: candidate mAP50={candidate['mAP50']:.4f}, val FPS={candidate['validation_pipeline_fps']:.1f}; no paired baseline available")
    weakest = sorted(candidate.get('per_class_AP50', {}).items(), key=lambda item: item[1])[:5]
    if weakest:
        print('    Lowest candidate class AP50:', ', '.join(f'{name}={score:.3f}' for name, score in weakest))

EXPORT_WEIGHTS = EXPORT_DIR / f'{RUN_NAME}_best.pt'
if EXPORT_WEIGHTS.exists():
    raise FileExistsError(f'Refusing to overwrite an existing export: {EXPORT_WEIGHTS}')
shutil.copy2(BEST_WEIGHTS, EXPORT_WEIGHTS)
REPORT_PATH = EXPORT_DIR / f'{RUN_NAME}_validation.json'
REPORT_PATH.write_text(json.dumps({
    'run_name': RUN_NAME,
    'dataset_split': 'val',
    'candidate_weights': str(EXPORT_WEIGHTS),
    'baseline_weights': str(BASELINE_WEIGHTS) if BASELINE_RESULTS else None,
    'baseline': BASELINE_RESULTS,
    'candidate': CANDIDATE_RESULTS,
}, indent=2), encoding='utf-8')
print('\nCandidate exported to:', EXPORT_WEIGHTS)
print('Validation report:', REPORT_PATH)


In [ ]:
# Change to True only after selecting/finalizing the candidate from validation.
# This evaluates the frozen baseline and candidate on test; do not use these results to tune.
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    if not BASELINE_WEIGHTS.is_file():
        raise FileNotFoundError(f'Paired final test requires the baseline checkpoint at {BASELINE_WEIGHTS}')
    FINAL_TEST_RESULTS = {
        'baseline': evaluate_checkpoint(BASELINE_WEIGHTS, image_sizes=(960,), split='test'),
        'candidate': evaluate_checkpoint(EXPORT_WEIGHTS, image_sizes=(960,), split='test'),
    }
    test_report = EXPORT_DIR / f'{RUN_NAME}_final_test.json'
    test_report.write_text(json.dumps(FINAL_TEST_RESULTS, indent=2), encoding='utf-8')
    print('Paired final test report:', test_report)
else:
    print('Final test not run. Set RUN_FINAL_TEST=True only after model selection is frozen.')


## After the run

- Inspect the training plots, `results.csv`, and saved validation JSON in `MyDrive/astra_adas_training/`. The `runs/<run-name>/weights/` folder contains a checkpoint for each epoch (`epoch*.pt`) plus `last.pt` and `best.pt`, saved to Drive.
- Prefer the candidate only if the paired full-validation comparison supports it, including per-class behavior and the 640/768/960 accuracy-throughput tradeoff. No target mAP is guaranteed.
- The test cell is disabled by default. After selecting and freezing the model from validation, enable it once for the paired held-out test report; do not tune from that report.
- Download the timestamped `*_best.pt` candidate and validation report. Back up `models/detection/best.pt` before any local replacement, then run `python -m src.evaluation.benchmark_m4 --split test --num-images all --imgsz 960` and inspect the report. Keep the candidate separate if results regress.
- This detector training does not train traffic-light state, traffic-sign types, lane detection, calibrated distance/speed, tracking, recommendations, or feedback-loop components. Continue those checklist items independently.

- Validation FPS is measured at batch size 1 on the Colab GPU and covers the detector validation stages only. It excludes video decoding, tracking, lane/traffic processing, drawing, and dashboard I/O. Benchmark the exported model on the target local hardware before claiming dashboard FPS.
